# Topología Aplicada y Computacional

## Práctica 3

Para el desarrollo de esta práctica se hará uso de los módulos `Simplex`, `SimplicialComplex`, `FilteredSimplicialComplex` y `AlphaFiltration` definidos en los laboratorios anteriores.

In [2]:
from homology.simplex import Simplex
from homology.simplicial_complex import SimplicialComplex
from homology.filtered_simplicial_complex import FilteredSimplicialComplex
from homology.filtrations.alpha_filtration import AlphaFiltration

**Exercise 1:** Definir un método dentro de la clase complejos simpliciales que calcule para cada p la matriz borde $M_{\partial_p}$.

En lugar de crear un método en la clase `SimplicialComplex`, se ha creado la clase *wrapper* `MatrixSimplicialComplex`, que se inicializa con un complejo simplical. Esto se hace tanto para seguir el principio de *separation of concerns* como para poder acelerar sus distintas funciones usando cachés.

Una vez provista de un complejo simplicial, la clase `SimplicialComplex` inicializa dos cachés:

- `_n_faces`: guarda los símplices de las llamadas a `SimplicialComplex.n_faces`, para no tener que recomputarlos.
- `_n_delta`: guarda las matrices de las llamadas a `MatrixSimplicialComplex.boundary_matrix`, para no tener que recomputarlas.

De este modo, el método `n_faces` busca las caras de dimensión $n$ en la caché y si no las encuentra las computa con el método de `SimplicialComplex`, y `boundary_matrix` busca la matriz de borde de la dimensión $n$ dada en la caché y si no la encuentra la computa usando los símplices de dimensión $n$ y $n-1$ del complejo simplicial.

In [6]:
from __future__ import annotations

import numpy as np

class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        
    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

Comprobamos el funcionamiento de la función calculando la matriz borde de dimensión 2 de un tetraedro:

In [9]:
sc = SimplicialComplex([Simplex((1,2,3,4))])
MatrixSimplicialComplex(sc).boundary_matrix(2)

array([[1, 1, 0, 0],
       [1, 0, 1, 0],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [0, 1, 0, 1],
       [0, 0, 1, 1]], dtype=uint8)

**Exercise 2:** Definir una función que calcule la forma normal de Smith de una matriz con coeficientes en $\mathbb{Z}_2$.

Como la forma normal de Smith en $\mathbb{Z}_2$ está completamente determinada por el rango de su matriz de borde correspondiente, se crea una nueva caché `_n_rank`. De este modo, cada llamada a `smith_boundary_matrix` comprueba si se conoce el rango de la matriz. Si se conoce, se reconstruye la matriz de forma eficiente. En caso contrario, se computa su rango usando un algoritmo de eliminación de filas y columnas.

In [10]:
class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        self._n_rank: dict[int, int] = {}

    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

    def rank_boundary_matrix(self, dim: int) -> int:
        """Return the rank of the boundary matrix over GF(2)."""
        if dim not in self._n_rank:
            mat = self.boundary_matrix(dim).copy()
            rows, cols = mat.shape

            rank = min(rows, cols)

            for k in range(min(rows, cols)):
                # Find a pivot in the remaining submatrix.
                nz = np.argwhere(mat[k:, k:] != 0)

                if nz.size == 0:
                    rank = k
                    break

                # Move the pivot to (k, k).
                i, j = nz[0] + k

                mat[[k, i], :] = mat[[i, k], :]
                mat[:, [k, j]] = mat[:, [j, k]]

                # Clear the rest of the pivot row and column.
                for nzcol in np.flatnonzero(mat[k, k+1:]) + k+1:
                    mat[:, nzcol] ^= mat[:, k]

                for nzrow in np.flatnonzero(mat[k+1:, k]) + k+1:
                    mat[nzrow, :] ^= mat[k, :]

            self._n_rank[dim] = rank

        return self._n_rank[dim]

    def smith_boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the Smith normal form over GF(2)."""
        rank = self.rank_boundary_matrix(dim)
        mat = np.zeros(self.boundary_matrix(dim).shape, dtype=np.uint8)

        for k in range(rank):
            mat[k, k] = 1

        return mat

Comprobamos el funcionamiento de la función calculando la forma normal de Smith de la matriz anterior:

In [11]:
sc = SimplicialComplex([Simplex((1,2,3,4))])
msc = MatrixSimplicialComplex(sc)
msc.smith_boundary_matrix(2)

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]], dtype=uint8)

**Exercise 3:** Definir un método que calcule los números de Betti de un complejo simplicial.

Una vez completados los dos ejercicios anteriores, obtener el número de Betti es trivial. Como $\beta_p(\sigma) = Z_p(\sigma) - B_p(\sigma)$, basta con obtener las matrices de borde en forma normal de Smith de dimensión $p$ y dimensión $p+1$ para conseguir $Z_p(\sigma)$ y $B_p(\sigma)$, respectivamente.

In [13]:
class MatrixSimplicialComplex:
    """Compute boundary matrices and Betti numbers over GF(2)."""

    def __init__(self, sc: SimplicialComplex) -> None:
        """Initialize the complex and its caches."""
        self._sc: SimplicialComplex = sc

        # The wrapped complex is assumed not to change after construction.
        self._n_faces: dict[int, list[Simplex]] = {}
        self._n_delta: dict[int, np.ndarray] = {}
        self._n_rank: dict[int, int] = {}

    def n_faces(self, dim: int) -> list[Simplex]:
        """Return the simplices of the given dimension."""
        if dim < -1:
            raise ValueError("Dimension must be at least -1")
        if dim == -1 or dim > self._sc.dim:
            return []

        if dim not in self._n_faces:
            self._n_faces[dim] = sorted(
                self._sc.n_faces(dim),
                key=lambda simplex: simplex.vertices,
            )

        return self._n_faces[dim]

    def boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the mod-2 boundary matrix from dimension dim to dim - 1."""
        if dim not in self._n_delta:
            row_faces = self.n_faces(dim - 1)
            col_faces = self.n_faces(dim)

            mat = np.zeros((len(row_faces), len(col_faces)), dtype=np.uint8)

            # Rows are (dim - 1)-faces and columns are dim-faces.
            for i, row_face in enumerate(row_faces):
                for j, col_face in enumerate(col_faces):
                    if row_face.is_face(col_face):
                        mat[i, j] = 1

            self._n_delta[dim] = mat

        return self._n_delta[dim]

    def rank_boundary_matrix(self, dim: int) -> int:
        """Return the rank of the boundary matrix over GF(2)."""
        if dim not in self._n_rank:
            mat = self.boundary_matrix(dim).copy()
            rows, cols = mat.shape

            rank = min(rows, cols)

            for k in range(min(rows, cols)):
                # Find a pivot in the remaining submatrix.
                nz = np.argwhere(mat[k:, k:] != 0)

                if nz.size == 0:
                    rank = k
                    break

                # Move the pivot to (k, k).
                i, j = nz[0] + k

                mat[[k, i], :] = mat[[i, k], :]
                mat[:, [k, j]] = mat[:, [j, k]]

                # Clear the rest of the pivot row and column.
                for nzcol in np.flatnonzero(mat[k, k+1:]) + k+1:
                    mat[:, nzcol] ^= mat[:, k]

                for nzrow in np.flatnonzero(mat[k+1:, k]) + k+1:
                    mat[nzrow, :] ^= mat[k, :]

            self._n_rank[dim] = rank

        return self._n_rank[dim]

    def smith_boundary_matrix(self, dim: int) -> np.ndarray:
        """Return the Smith normal form over GF(2)."""
        rank = self.rank_boundary_matrix(dim)
        mat = np.zeros(self.boundary_matrix(dim).shape, dtype=np.uint8)

        for k in range(rank):
            mat[k, k] = 1

        return mat

    def boundary_space_dim(self, k: int) -> int:
        """Return the dimension of the boundary space B_k."""
        return self.rank_boundary_matrix(k + 1)

    def cycle_space_dim(self, k: int) -> int:
        """Return the dimension of the cycle space Z_k."""
        return len(self.n_faces(k)) - self.rank_boundary_matrix(k)

    def betti_number(self, k: int) -> int:
        """Return the k-th Betti number over GF(2)."""
        if k < 0:
            raise ValueError("Betti numbers are only defined for k >= 0")

        return self.cycle_space_dim(k) - self.boundary_space_dim(k)

Una vez más, comprobamos que funciona con el tetraedro:

In [14]:
sc = SimplicialComplex([Simplex((0, 1, 2, 3))])
msc = MatrixSimplicialComplex(sc)

for i in range(4):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0
betti(3) = 0


**Exercise 4:** Calcular los números de Betti del tetraedro.

Para facilitar los ejercicios siguientes, se crea la *factory class* `SimplicialComplexFactory`, para instanciar distintos ejemplos de complejos simpliciales.

In [40]:
class SimplicialComplexFactory:
    """Factory for standard simplicial complexes used in examples and tests."""

    @staticmethod
    def n_simplicial(n: int) -> SimplicialComplex:
        if n < 0:
            raise ValueError("Dimension must be non-negative")
            
        """Return the (n - 1)-simplex on the vertices 0 to n - 1."""
        return SimplicialComplex([Simplex(tuple(range(n)))])

    @staticmethod
    def torus() -> SimplicialComplex:
        """Return a triangulation of the torus."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4), (2,4,5), (2,3,5), (3,5,6), (1,3,6), (1,4,6),
                (4,5,7), (5,7,8), (5,6,8), (6,8,9), (4,6,9), (4,7,9),
                (1,7,8), (1,2,8), (2,8,9), (2,3,9), (3,7,9), (1,3,7)
            ]
        ))

    @staticmethod
    def torus_minimal() -> SimplicialComplex:
        """Return a minimal triangulation of the torus."""
        return SimplicialComplex(map(Simplex, [
                (1,3,7), (1,2,3), (2,3,6), (3,4,6), (4,6,7), (1,6,7), (2,5,6),
                (1,5,6), (2,5,7), (3,5,7), (1,4,5), (3,4,5), (1,2,4), (2,4,7)
            ]
        ))

    @staticmethod
    def klein_bottle() -> SimplicialComplex:
        """Return a triangulation of the Klein bottle."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4), (2,4,5), (2,3,5), (3,5,6), (1,3,6), (1,7,6),
                (4,5,7), (5,7,8), (5,6,8), (6,8,9), (4,6,9), (4,7,9),
                (1,7,8), (1,2,8), (2,8,9), (2,3,9), (3,7,9), (1,4,3)
            ]
        ))

    @staticmethod
    def ring() -> SimplicialComplex:
        """Return a triangulation of the annulus."""
        return SimplicialComplex(map(Simplex, [
                (1,2,4),(1,3,6),(1,4,6),
                (2,3,5),(2,4,5),(3,5,6)
            ]
        ))

    @staticmethod
    def projective_plane() -> SimplicialComplex:
        """Return a triangulation of the projective plane."""
        return SimplicialComplex(map(Simplex, [
                (1,2,6), (2,3,4), (1,3,4), (1,2,5), (2,3,5),
                (1,3,6), (2,4,6), (1,4,5), (3,5,6), (4,5,6)
            ]
        ))

    @staticmethod
    def double_torus() -> SimplicialComplex:
        """Return a triangulation of the double torus."""
        return SimplicialComplex(map(Simplex, [
                (0, 1, 3), (0, 1, 5), (0, 2, 3), (0, 2, 6), (0, 4, 5), (0, 4, 9),
                (0, 6, 8), (0, 7, 8), (0, 7, 9), (1, 2, 4), (1, 2, 6), (1, 3, 4),
                (1, 5, 6), (2, 3, 5), (2, 4, 5), (3, 4, 9), (3, 5, 8), (3, 6, 7),
                (3, 6, 9), (3, 7, 8), (5, 6, 7), (5, 7, 9), (5, 8, 9), (6, 8, 9),
            ]
        ))

    @staticmethod
    def dunce_hat() -> SimplicialComplex:
        """Return a triangulation of the dunce hat."""
        return SimplicialComplex(map(Simplex, [
                (1,3,5), (1,5,6), (1,3,6), (2,4,5),
                (4,5,6), (4,6,8), (6,7,8), (3,6,7),
                (3,2,7), (1,2,7), (1,7,8), (1,2,4),
                (1,3,4), (3,4,8), (2,3,8), (1,2,8)
            ]
        ))

    @staticmethod
    def example() -> SimplicialComplex:
        """Return a mixed example complex with simplices of different dimensions."""
        return SimplicialComplex(map(Simplex, [
                (0, 1),
                (1, 2, 3, 4),
                (4, 5), (5, 6), (4, 6),
                (6, 7, 8),
                (8, 9)
            ]
        ))

    @staticmethod
    def alpha_complex_example_1() -> SimplicialComplex:
        """Return the alpha complex of the first point cloud at radius 0.2."""
        r = 0.20
        points = [
            (0.38021546727456423, 0.4641920233959878), (0.7951628297672293, 0.4926363013586947),
            (0.566623772375203, 0.038325621649018426), (0.3369306814864865, 0.7103735061134965),
            (0.08272837815822842, 0.2263273314352896), (0.5180166301873989, 0.6271769943824689),
            (0.33691411899985035, 0.8402045183219995), (0.33244488399729255, 0.452463652047520),
            (0.11778991601260325, 0.6657734204021165), (0.9384303415747769, 0.2313873874340855)
        ]

        alpha_filtration = AlphaFiltration(points).build()

        return alpha_filtration.filtration(r)

    @staticmethod
    def alpha_complex_example_2() -> SimplicialComplex:
        """Return the alpha complex of the first point cloud at radius 1.0."""
        r = 1.00
        points = [
            (0.38021546727456423, 0.4641920233959878), (0.7951628297672293, 0.4926363013586947),
            (0.566623772375203, 0.038325621649018426), (0.3369306814864865, 0.7103735061134965),
            (0.08272837815822842, 0.2263273314352896), (0.5180166301873989, 0.6271769943824689),
            (0.33691411899985035, 0.8402045183219995), (0.33244488399729255, 0.452463652047520),
            (0.11778991601260325, 0.6657734204021165), (0.9384303415747769, 0.2313873874340855)
        ]

        alpha_filtration = AlphaFiltration(points).build()

        return alpha_filtration.filtration(r)

Probamos el tetraedro:

In [41]:
sc = SimplicialComplexFactory.n_simplicial(4)
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0
betti(3) = 0


**Exercise 5:** Calcular los números de Betti del borde del tetraedro.

In [42]:
sc = SimplicialComplex(
    SimplicialComplexFactory.n_simplicial(4).skeleton(2)
)

msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 1


**Exercise 6:** Calcular los números de Betti del toro con las dos triangulaciones que hemos visto en clase.

Comprobamos primero una triangulación normal:

In [43]:
sc = SimplicialComplexFactory.torus()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 2
betti(2) = 1


Y a continuación probamos la triangulación mínima:

In [44]:
sc = SimplicialComplexFactory.torus_minimal()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 2
betti(2) = 1


**Exercise 7:** Calcular los números de Betti de la botella de Klein.

In [45]:
sc = SimplicialComplexFactory.klein_bottle()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 3
betti(2) = 0


**Exercise 8:** Calcular los números de Betti del anillo.

In [46]:
sc = SimplicialComplexFactory.ring()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0


**Exercise 9:** Calcular los números de Betti del plano proyectivo.

In [47]:
sc = SimplicialComplexFactory.projective_plane()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 1


**Exercise 10:** Calcular los números de Betti del sombrero del asno.

In [48]:
sc = SimplicialComplexFactory.dunce_hat()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0


**Exercise 11:** Calcular los números de Betti del doble toro.

In [49]:
sc = SimplicialComplexFactory.double_torus()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 4
betti(2) = 1


**Exercise 12:** Calcular los números de Betti del complejo simplicial de la transparencia 4.

In [50]:
sc = SimplicialComplexFactory.example()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 1
betti(2) = 0
betti(3) = 0


**Exercise 13:** Calcular los números de Betti de algunos alfa complejos.

Comprobamos los números de Betti de un alfa complejo al variar el instante:

In [52]:
sc = SimplicialComplexFactory.alpha_complex_example_1()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 2
betti(1) = 0
betti(2) = 0


In [53]:
sc = SimplicialComplexFactory.alpha_complex_example_2()
msc = MatrixSimplicialComplex(sc)

for i in range(sc.dim+1):
    print(f"betti({i}) = {msc.betti_number(i)}")

betti(0) = 1
betti(1) = 0
betti(2) = 0
